In [1]:
import osmnx as ox
import geopandas as gpd
from pathlib import Path

RAW_DIR = Path("../../data/raw/05_red_viaria")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Descargando red viaria para camiones en Huesca...")

G = ox.graph_from_place(
    "Huesca, Spain",
    network_type="drive",        # solo vías transitables por vehículo
    simplify=True,
    custom_filter='["highway"~"motorway|motorway_link|trunk|trunk_link|primary|primary_link|secondary|secondary_link|tertiary|tertiary_link"]'
)

nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True)

print(f"Nodos:  {len(nodes):,}")
print(f"Tramos: {len(edges):,}")

output_gpkg = RAW_DIR / "carreteras_camiones_huesca.gpkg"
edges.to_file(output_gpkg, layer="edges", driver="GPKG")
nodes.to_file(output_gpkg, layer="nodes", driver="GPKG")
print(f"Guardado: {output_gpkg}")

print("\nTipos de vía:")
print(edges["highway"].explode().value_counts().to_string())

Descargando red viaria para camiones en Huesca...


Nodos:  4,881
Tramos: 9,373


Guardado: ../../data/raw/05_red_viaria/carreteras_camiones_huesca.gpkg

Tipos de vía:
highway
tertiary          2325
trunk             1945
secondary         1677
primary           1380
trunk_link         470
tertiary_link      421
secondary_link     373
primary_link       329
motorway_link      291
motorway           198


In [2]:
import numpy as np
import rasterio
import geopandas as gpd
import pandas as pd
import pydeck as pdk
from pyproj import Transformer
from shapely.geometry import mapping
import json
from pathlib import Path

RAW_DIR = Path("../../data/raw/05_red_viaria")
MAP_DIR = Path("../../data/map/05_red_viaria")
SLOPE_FILE = Path("../../data/raw/03_pendiente_dem/pendiente_huesca_provincia.tif")
RAW_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

# --- 1. Leer pendiente ---
print("Leyendo pendiente...")
with rasterio.open(SLOPE_FILE) as src:
    slope = src.read(1).astype("float32")
    nodata = src.nodata or -9999
    slope[slope == nodata] = np.nan
    transform = src.transform

step = 25
slope_sub = slope[::step, ::step]
rows, cols = slope_sub.shape

xs = np.array([transform.c + (j * step + step/2) * transform.a for j in range(cols)])
ys = np.array([transform.f + (i * step + step/2) * transform.e for i in range(rows)])
X, Y = np.meshgrid(xs, ys)

mask = ~np.isnan(slope_sub)
x_flat, y_flat, s_flat = X[mask], Y[mask], slope_sub[mask]

tr = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)
lon, lat = tr.transform(x_flat, y_flat)

df_slope = pd.DataFrame({
    "lon": lon,
    "lat": lat,
    "pendiente": np.round(s_flat, 1),
})
print(f"Puntos pendiente: {len(df_slope):,}")

# --- 2. Leer carreteras ---
print("Leyendo carreteras...")
edges = gpd.read_file(RAW_DIR / "carreteras_camiones_huesca.gpkg", layer="edges").to_crs("EPSG:4326")

color_map = {
    "motorway":  [ 20,  20,  20, 255],
    "trunk":     [ 50,  50,  50, 255],
    "primary":   [ 80,  80,  80, 255],
    "secondary": [110, 110, 110, 255],
    "tertiary":  [140, 140, 140, 255],
}
DEFAULT_COLOR = [100, 100, 100, 200]

def get_color(hw):
    if isinstance(hw, list): hw = hw[0]
    for key, val in color_map.items():
        if isinstance(hw, str) and key in hw:
            return val
    return DEFAULT_COLOR

def get_width(hw):
    if isinstance(hw, list): hw = hw[0]
    if isinstance(hw, str):
        if "motorway" in hw or "trunk" in hw: return 150
        if "primary" in hw:                   return 110
        if "secondary" in hw:                 return 75
    return 45

edges["color"] = edges["highway"].apply(get_color)
edges["width"] = edges["highway"].apply(get_width)

features = []
for _, row in edges.iterrows():
    hw = row["highway"]
    if isinstance(hw, list): hw = hw[0]
    tipo = {
        "motorway":  "Autopista",
        "trunk":     "Vía rápida",
        "primary":   "Carretera nacional",
        "secondary": "Carretera comarcal",
        "tertiary":  "Carretera local",
    }.get(str(hw).split("_")[0], str(hw))

    features.append({
        "type": "Feature",
        "geometry": mapping(row.geometry),
        "properties": {
            "color":      row["color"],
            "width":      row["width"],
            "width_halo": row["width"] + 60,
            "tipo_via":   tipo,
            "nombre":     str(row.get("name", "—") or "—"),
        }
    })
geojson_roads = {"type": "FeatureCollection", "features": features}

# --- 3. Capas ---

# Heatmap de pendiente
layer_slope = pdk.Layer(
    "HeatmapLayer",
    data=df_slope,
    get_position=["lon", "lat"],
    get_weight="pendiente",
    aggregation="MEAN",
    radius_pixels=25,
    intensity=1,
    threshold=0.05,
    color_range=[
        [ 34, 139,  34, 180],   # verde oscuro  — plano
        [144, 238, 144, 190],   # verde claro   — suave
        [220, 220,  50, 210],   # amarillo      — moderado
        [255, 140,   0, 220],   # naranja       — fuerte
        [200,  30,   0, 230],   # rojo          — extremo
    ],
    pickable=False,
)

# Halo negro debajo de las carreteras
layer_roads_halo = pdk.Layer(
    "GeoJsonLayer",
    data=geojson_roads,
    stroked=True,
    filled=False,
    get_line_color=[0, 0, 0, 220],
    get_line_width="properties.width_halo",
    line_width_min_pixels=4,
    pickable=False,
)

# Trazo real de carreteras encima
layer_roads = pdk.Layer(
    "GeoJsonLayer",
    data=geojson_roads,
    stroked=True,
    filled=False,
    get_line_color="properties.color",
    get_line_width="properties.width",
    line_width_min_pixels=2,
    pickable=True,
)

view = pdk.ViewState(
    longitude=-0.08,
    latitude=42.10,
    zoom=8,
    min_zoom=7,
    max_zoom=13,
    pitch=0,
    bearing=0,
)

tooltip = {
    "html": "<b>{tipo_via}</b><br/>{nombre}",
    "style": {"backgroundColor": "#111", "color": "white",
              "fontSize": "12px", "padding": "6px"},
}

deck = pdk.Deck(
    layers=[layer_slope, layer_roads_halo, layer_roads],
    initial_view_state=view,
    tooltip=tooltip,
    map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
)

leyenda_html = """
<div style="position:absolute; bottom:30px; left:20px; background:rgba(0,0,0,0.75);
            padding:14px 18px; border-radius:8px; font-family:sans-serif;
            font-size:13px; color:white; z-index:999; line-height:1.9;">

  <div style="font-weight:bold; font-size:14px; margin-bottom:8px;">🗺 Leyenda</div>

  <div style="margin-bottom:6px; font-weight:bold; color:#aaa;">Pendiente del terreno</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#228B22;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>0–5° Plano</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#90EE90;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>5–15° Suave</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#DCDC32;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>15–30° Moderado</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#FF8C00;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>30–45° Fuerte</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#C81E00;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>&gt;45° Muy fuerte</div>

  <div style="margin:10px 0 6px; font-weight:bold; color:#aaa;">Red viaria</div>
  <div><span style="display:inline-block;width:28px;height:4px;background:#141414;border-radius:2px;margin-right:8px;vertical-align:middle;"></span>Autopista</div>
  <div><span style="display:inline-block;width:28px;height:4px;background:#323232;border-radius:2px;margin-right:8px;vertical-align:middle;"></span>Vía rápida</div>
  <div><span style="display:inline-block;width:28px;height:4px;background:#505050;border-radius:2px;margin-right:8px;vertical-align:middle;"></span>Carretera nacional</div>
  <div><span style="display:inline-block;width:28px;height:4px;background:#6e6e6e;border-radius:2px;margin-right:8px;vertical-align:middle;"></span>Carretera comarcal</div>
  <div><span style="display:inline-block;width:28px;height:4px;background:#8c8c8c;border-radius:2px;margin-right:8px;vertical-align:middle;"></span>Carretera local</div>
</div>
"""

html_output = deck.to_html(as_string=True)
html_output = html_output.replace("</body>", leyenda_html + "</body>")

with open(MAP_DIR / "mapa_pendiente_carreteras_huesca.html", "w", encoding="utf-8") as f:
    f.write(html_output)

print(f"Guardado: {MAP_DIR / 'mapa_pendiente_carreteras_huesca.html'}")

Leyendo pendiente...


Puntos pendiente: 40,125
Leyendo carreteras...


Guardado: ../../data/map/05_red_viaria/mapa_pendiente_carreteras_huesca.html
